# Agent 1: Symptom Classifier — Evaluation

Evaluate the fine-tuned LLaMA-3.2-3B-Instruct QLoRA adapter on test data.

**Metrics:**
- Department accuracy
- Urgency accuracy
- Overall accuracy (both correct)
- Per-department & per-urgency breakdown

In [1]:
!pip install -q transformers>=4.45.0 peft>=0.13.0 bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets huggingface_hub

In [2]:
import shutil, os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    if os.path.exists('/content/drive') and os.listdir('/content/drive'):
        shutil.rmtree('/content/drive')
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

Mounted at /content/drive


In [3]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in to HuggingFace via Colab Secrets')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')
    if hf_token:
        login(token=hf_token)
        print('Logged in via env var')
    else:
        print('WARNING: No HF_TOKEN found!')

Logged in to HuggingFace via Colab Secrets


In [8]:
import torch

# ============================================================
# CONFIG
# ============================================================
MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER_DIR = '/content/drive/MyDrive/symptom_classifier_adapter/final_adapter'
TEST_FILE = '/content/drive/MyDrive/symptom_classifier_test.jsonl'
EVAL_LIMIT = None  # Set to e.g. 100 for quick test, None for full eval

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected!')

GPU: NVIDIA L4 | VRAM: 23.7 GB


In [9]:
import json, random

with open(TEST_FILE) as f:
    test_data = [json.loads(line) for line in f]

if EVAL_LIMIT:
    random.seed(42)
    test_data = random.sample(test_data, min(EVAL_LIMIT, len(test_data)))

print(f'Loaded {len(test_data)} test samples')
print(f'Sample keys: {list(test_data[0].keys())}')
print(f'Sample input: {test_data[0]["input"][:150]}...')

Loaded 738 test samples
Sample keys: ['instruction', 'input', 'output', 'disease', 'department', 'urgency']
Sample input: Patient reports: itching, fatigue, lethargy, yellowish skin, dark urine, loss of appetite, yellow urine, yellowing of eyes, malaise, receiving blood t...


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print('Model loaded successfully.')

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully.


In [11]:
from collections import defaultdict

SYSTEM_PROMPT = (
    'You are a medical triage assistant. Classify the patient\'s symptoms '
    'into a medical department and urgency level. Respond in exactly this format:\n'
    'Department: <department>\nUrgency: <Routine|Urgent|Emergency>'
)

correct_dept, correct_urg, correct_both = 0, 0, 0

# Per-class tracking
dept_tp = defaultdict(int)
dept_fp = defaultdict(int)
dept_fn = defaultdict(int)
urg_tp = defaultdict(int)
urg_fp = defaultdict(int)
urg_fn = defaultdict(int)

errors = []  # Store misclassifications for analysis

for i, sample in enumerate(test_data):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': sample['input']},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    input_len = tokenized['input_ids'].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **tokenized, max_new_tokens=50, temperature=0.1,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    # Parse prediction
    pred_dept, pred_urg = 'General Medicine', 'Routine'
    for line in generated.split('\n'):
        if line.startswith('Department:'):
            pred_dept = line.replace('Department:', '').strip()
        elif line.startswith('Urgency:'):
            pred_urg = line.replace('Urgency:', '').strip()

    true_dept = sample['department']
    true_urg = sample['urgency']

    # Department metrics
    if pred_dept == true_dept:
        dept_tp[true_dept] += 1
        correct_dept += 1
    else:
        dept_fp[pred_dept] += 1
        dept_fn[true_dept] += 1
        errors.append({'input': sample['input'][:100], 'true': f'{true_dept}/{true_urg}', 'pred': f'{pred_dept}/{pred_urg}'})

    # Urgency metrics
    if pred_urg == true_urg:
        urg_tp[true_urg] += 1
        correct_urg += 1
    else:
        urg_fp[pred_urg] += 1
        urg_fn[true_urg] += 1

    if pred_dept == true_dept and pred_urg == true_urg:
        correct_both += 1

    if (i + 1) % 20 == 0:
        print(f'  [{i+1}/{len(test_data)}] accuracy: {correct_both/(i+1):.1%}')

print(f'\nEvaluation complete. Processed {len(test_data)} samples.')

  [20/738] accuracy: 95.0%
  [40/738] accuracy: 92.5%
  [60/738] accuracy: 93.3%
  [80/738] accuracy: 92.5%
  [100/738] accuracy: 94.0%
  [120/738] accuracy: 95.0%
  [140/738] accuracy: 95.0%
  [160/738] accuracy: 95.0%
  [180/738] accuracy: 95.0%
  [200/738] accuracy: 94.5%
  [220/738] accuracy: 95.0%
  [240/738] accuracy: 95.4%
  [260/738] accuracy: 95.4%
  [280/738] accuracy: 95.4%
  [300/738] accuracy: 95.7%
  [320/738] accuracy: 95.9%
  [340/738] accuracy: 96.2%
  [360/738] accuracy: 96.1%
  [380/738] accuracy: 96.1%
  [400/738] accuracy: 96.2%
  [420/738] accuracy: 96.4%
  [440/738] accuracy: 96.1%
  [460/738] accuracy: 96.3%
  [480/738] accuracy: 96.5%
  [500/738] accuracy: 96.4%
  [520/738] accuracy: 96.2%
  [540/738] accuracy: 96.3%
  [560/738] accuracy: 96.2%
  [580/738] accuracy: 96.0%
  [600/738] accuracy: 96.0%
  [620/738] accuracy: 95.8%
  [640/738] accuracy: 95.9%
  [660/738] accuracy: 96.1%
  [680/738] accuracy: 96.2%
  [700/738] accuracy: 96.3%
  [720/738] accuracy: 96

In [12]:
def compute_prf(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

n = len(test_data)

print('=' * 60)
print('SYMPTOM CLASSIFIER EVALUATION REPORT')
print('=' * 60)
print(f'Total samples:       {n}')
print(f'Overall accuracy:    {correct_both/n:.1%}')
print(f'Department accuracy: {correct_dept/n:.1%}')
print(f'Urgency accuracy:    {correct_urg/n:.1%}')

# Emergency recall
em_recall = urg_tp['Emergency'] / (urg_tp['Emergency'] + urg_fn['Emergency']) if (urg_tp['Emergency'] + urg_fn['Emergency']) > 0 else 0.0
print(f'Emergency recall:    {em_recall:.1%}  (target: >95%)')

# Per-Department
print(f'\n--- Per-Department ---')
print(f'{"Department":<22} {"Prec":>6} {"Recall":>6} {"F1":>6} {"Support":>8}')
all_depts = sorted(set(list(dept_tp) + list(dept_fn)))
for d in all_depts:
    p, r, f = compute_prf(dept_tp[d], dept_fp[d], dept_fn[d])
    support = dept_tp[d] + dept_fn[d]
    print(f'{d:<22} {p:>6.1%} {r:>6.1%} {f:>6.1%} {support:>8}')

# Per-Urgency
print(f'\n--- Per-Urgency ---')
print(f'{"Urgency":<22} {"Prec":>6} {"Recall":>6} {"F1":>6} {"Support":>8}')
all_urgs = sorted(set(list(urg_tp) + list(urg_fn)))
for u in all_urgs:
    p, r, f = compute_prf(urg_tp[u], urg_fp[u], urg_fn[u])
    support = urg_tp[u] + urg_fn[u]
    print(f'{u:<22} {p:>6.1%} {r:>6.1%} {f:>6.1%} {support:>8}')

SYMPTOM CLASSIFIER EVALUATION REPORT
Total samples:       738
Overall accuracy:    96.5%
Department accuracy: 99.7%
Urgency accuracy:    96.7%
Emergency recall:    99.4%  (target: >95%)

--- Per-Department ---
Department               Prec Recall     F1  Support
Cardiology              98.3% 100.0%  99.2%       59
Dermatology            100.0% 100.0% 100.0%      119
Endocrinology          100.0%  98.6%  99.3%       74
Gastroenterology       100.0% 100.0% 100.0%      204
General Medicine       100.0% 100.0% 100.0%       19
Infectious Disease     100.0%  98.7%  99.3%       75
Neurology              100.0% 100.0% 100.0%       71
Orthopedics            100.0% 100.0% 100.0%       39
Pulmonology            100.0% 100.0% 100.0%       59
Urology                 95.0% 100.0%  97.4%       19

--- Per-Urgency ---
Urgency                  Prec Recall     F1  Support
Emergency              100.0%  99.4%  99.7%      156
Routine                 97.8%  98.9%  98.4%      181
Urgent                  99.

In [13]:
# Show some misclassifications for error analysis
print(f'\n--- Error Analysis (showing first 10 of {len(errors)} errors) ---')
for e in errors[:10]:
    print(f'  Input: {e["input"]}...')
    print(f'  True: {e["true"]}  |  Pred: {e["pred"]}')
    print()


--- Error Analysis (showing first 10 of 2 errors) ---
  Input: Patient reports: vomiting, fatigue, sweating, headache, nausea, blurred and distorted vision, excess...
  True: Endocrinology/Urgent  |  Pred: Cardiology/Urgent

  Input: Patient reports: chills, vomiting, fatigue, high fever, headache, nausea, constipation, abdominal pa...
  True: Infectious Disease/Emergency  |  Pred: Urology/Emergency



In [14]:
# Save results to Drive
results = {
    'total': n,
    'overall_accuracy': correct_both / n,
    'department_accuracy': correct_dept / n,
    'urgency_accuracy': correct_urg / n,
    'emergency_recall': em_recall,
}

output_path = '/content/drive/MyDrive/eval_symptom_classifier_results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {output_path}')

Results saved to /content/drive/MyDrive/eval_symptom_classifier_results.json
